# Module 6 Guided Lab: String and Categorical Data Transformation

**BAN 6003: Data Management and Analytics Integration**

In this lab, we work with a small intentionally messy business dataset. The goal is to practice practical string and categorical transformations using Pandas.

This lab focuses on:

- identifying messy text and categorical values,
- standardizing casing, spacing, and labels,
- parsing simple combined text fields,
- recoding categories using clear business rules,
- writing short notes explaining recoding assumptions.

We will **not** do heavy validation, reshaping, or aggregation in this module. Those topics appear later.

## Lab Learning Outcomes

By the end of this lab, you should be able to:

1. Identify common text and categorical data issues in business datasets.
2. Use Pandas string methods such as `.str.strip()`, `.str.lower()`, `.str.upper()`, `.str.title()`, `.str.replace()`, `.str.contains()`, and `.str.split()`.
3. Recode messy categorical values into analysis-ready categories using `.replace()` and `.map()`.
4. Explain how recoding choices affect interpretation.


## Part 0: Setup

We will use Pandas and a small customer contact dataset. The dataset is intentionally messy, but small enough to inspect directly.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [2]:
raw_path = Path("../data/raw/messy_customer_contacts.csv")

if not raw_path.is_file():
    raise FileNotFoundError(f"Required raw input was not found: {raw_path}")

df_raw = pd.read_csv(raw_path)
df_raw.head(10)

,customer_id,customer_name,email,state_raw,segment_raw,contact_method_raw,channel_raw,product_raw,issue_text
0,1001,Alice Smith,alice.smith@example.com,TX,Premium,Email,web - organic,Loan_Application,Customer says login failed twice
1,1002,bob jones,bob.jones@example.com,Texas,VIP,e-mail,Web-Organic,loan application,Asked about mortgage rate
2,1003,CARLA NGUYEN,carla.nguyen@example.com,tx.,standard,phone,Mobile App,Checking,Reported unauthorized charge
3,1004,Daniel Lee,daniel.lee@example.com,CA,basic,Phone Call,branch,Credit_Card,Needs card replacement
4,1005,erin o'neill,erin.oneill@example.com,California,Standard,SMS,mobile-app,credit card,Asked to update address
5,1006,Fatima Khan,fatima.khan@example.com,ca,vip,text,WEB - PAID,Mortgage,Possible fraud on account
6,1007,george martin,george.martin@example.com,NY,Premium,EMAIL,web-paid,mortgage,Wants payoff estimate
7,1008,HANA KIM,hana.kim@example.com,New York,unknown,chat,Call Center,Auto_Loan,Asked about late fee
8,1009,ivan petrov,ivan.petrov@example.com,N.Y.,Basic,Chat,call center,auto loan,Fraud alert question
9,1010,Julia Brown,julia.brown@example.com,FL,standard,email,Partner Referral,Checking_Account,New account opening


## Part 1: First Look at Messy Text and Categories

Before changing text values, inspect them. Do not assume you know all possible categories.

We will use `.value_counts()` to look for:

- inconsistent capitalization,
- extra spaces,
- abbreviations,
- spelling differences,
- labels that may mean the same thing.


In [3]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   customer_id         15 non-null     int64
 1   customer_name       15 non-null     str  
 2   email               15 non-null     str  
 3   state_raw           15 non-null     str  
 4   segment_raw         15 non-null     str  
 5   contact_method_raw  15 non-null     str  
 6   channel_raw         15 non-null     str  
 7   product_raw         15 non-null     str  
 8   issue_text          15 non-null     str  
dtypes: int64(1), str(8)
memory usage: 2.6 KB


In [4]:
df_raw[["state_raw", "segment_raw", "contact_method_raw", "channel_raw", "product_raw"]].head(10)

,state_raw,segment_raw,contact_method_raw,channel_raw,product_raw
0,TX,Premium,Email,web - organic,Loan_Application
1,Texas,VIP,e-mail,Web-Organic,loan application
2,tx.,standard,phone,Mobile App,Checking
3,CA,basic,Phone Call,branch,Credit_Card
4,California,Standard,SMS,mobile-app,credit card
5,ca,vip,text,WEB - PAID,Mortgage
6,NY,Premium,EMAIL,web-paid,mortgage
7,New York,unknown,chat,Call Center,Auto_Loan
8,N.Y.,Basic,Chat,call center,auto loan
9,FL,standard,email,Partner Referral,Checking_Account


In [5]:
df_raw["state_raw"].value_counts(dropna=False)

state_raw
CA            2
TX            1
Texas         1
tx.           1
California    1
ca            1
NY            1
New York      1
N.Y.          1
FL            1
Florida       1
fl.           1
TX            1
Nevada        1
Name: count, dtype: int64

In [6]:
df_raw["contact_method_raw"].value_counts(dropna=False)

contact_method_raw
phone           2
EMAIL           2
Email           1
e-mail          1
Phone Call      1
SMS             1
text            1
chat            1
Chat            1
email           1
E-Mail          1
Text Message    1
sms             1
Name: count, dtype: int64

In [7]:
df_raw["product_raw"].value_counts(dropna=False)

product_raw
Loan_Application    1
loan application    1
Checking            1
Credit_Card         1
credit card         1
Mortgage            1
mortgage            1
Auto_Loan           1
auto loan           1
Checking_Account    1
checking account    1
Credit Card         1
Student_Loan        1
student loan        1
Personal_Loan       1
Name: count, dtype: int64

### Quick reflection

Write 2–3 short observations about the messy categories you see.

Example: `TX`, `Texas`, and `tx.` probably refer to the same state, but they appear as different values.


> Your notes here:
>
> - Checking_Account and Checking refer to the same type of account but appear as different values.
> - EMAIL, email, Email, and e-mail appear as different values but are the same.
> - text and SMS are different values meaning the same and should be standardized.

## Part 2: Keep the Raw Data and Create a Working Copy

A good habit is to keep raw columns unchanged and create cleaned columns next to them. This lets you compare before and after.

In [8]:
df = df_raw.copy()
df.head()

,customer_id,customer_name,email,state_raw,segment_raw,contact_method_raw,channel_raw,product_raw,issue_text
0,1001,Alice Smith,alice.smith@example.com,TX,Premium,Email,web - organic,Loan_Application,Customer says login failed twice
1,1002,bob jones,bob.jones@example.com,Texas,VIP,e-mail,Web-Organic,loan application,Asked about mortgage rate
2,1003,CARLA NGUYEN,carla.nguyen@example.com,tx.,standard,phone,Mobile App,Checking,Reported unauthorized charge
3,1004,Daniel Lee,daniel.lee@example.com,CA,basic,Phone Call,branch,Credit_Card,Needs card replacement
4,1005,erin o'neill,erin.oneill@example.com,California,Standard,SMS,mobile-app,credit card,Asked to update address


## Part 3: Basic String Standardization

Common string cleaning steps include:

- `.str.strip()` to remove extra spaces at the beginning or end,
- `.str.lower()` or `.str.upper()` to standardize casing,
- `.str.title()` to make names easier to read,
- `.str.replace()` to replace punctuation or repeated spacing.

Let's start with customer names.


In [9]:
df["customer_name_clean"] = (
    df["customer_name"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.title()
)

df[["customer_name", "customer_name_clean"]].head(10)

,customer_name,customer_name_clean
0,Alice Smith,Alice Smith
1,bob jones,Bob Jones
2,CARLA NGUYEN,Carla Nguyen
3,Daniel Lee,Daniel Lee
4,erin o'neill,Erin O'Neill
5,Fatima Khan,Fatima Khan
6,george martin,George Martin
7,HANA KIM,Hana Kim
8,ivan petrov,Ivan Petrov
9,Julia Brown,Julia Brown


Notice that we are not changing the underlying customer identity. We are making the text easier to read and more consistent.

## Part 4: Standardize State Labels

The `state_raw` column has values like `TX`, `Texas`, `tx.`, `N.Y.`, and `New York`. We need a consistent state code.

A common workflow is:

1. strip extra spaces,
2. convert to uppercase,
3. remove periods,
4. use a mapping dictionary to convert long names to standard codes.


In [21]:
df["state_step1"] = (
    df["state_raw"]
    .str.strip()
    .str.upper()
    .str.replace(".", "", regex=False)
)

df[["state_raw", "state_step1"]].drop_duplicates().sort_values("state_raw")

,state_raw,state_step1
3,CA,CA
4,California,CALIFORNIA
9,FL,FL
10,Florida,FLORIDA
8,N.Y.,NY
6,NY,NY
14,Nevada,NEVADA
7,New York,NEW YORK
12,TX,TX
0,TX,TX


In [11]:
state_map = {
    "TEXAS": "TX",
    "TX": "TX",
    "CALIFORNIA": "CA",
    "CA": "CA",
    "NEW YORK": "NY",
    "NY": "NY",
    "FLORIDA": "FL",
    "FL": "FL",
    "NEVADA": "NV"
}

df["state_clean"] = df["state_step1"].map(state_map)

df[["state_raw", "state_step1", "state_clean"]].drop_duplicates().sort_values("state_raw")

,state_raw,state_step1,state_clean
3,CA,CA,CA
4,California,CALIFORNIA,CA
9,FL,FL,FL
10,Florida,FLORIDA,FL
8,N.Y.,NY,NY
6,NY,NY,NY
14,Nevada,NEVADA,NV
7,New York,NEW YORK,NY
12,TX,TX,TX
0,TX,TX,TX


### Why mapping matters

The mapping dictionary is a business rule. It says which raw labels we believe are equivalent.

If a value does not appear in the mapping dictionary, Pandas will return a missing value. That is useful because it shows us which categories still need a decision.


## Part 5: Recode Contact Method Labels

The `contact_method_raw` column contains labels like `Email`, `e-mail`, `E-Mail`, `SMS`, `Text Message`, `Phone Call`, and `chat`.

We will standardize these into a smaller set of analysis-ready categories:

- `Email`
- `Phone`
- `Text`
- `Chat`


In [22]:
df["contact_method_step1"] = (
    df["contact_method_raw"]
    .str.strip()
    .str.lower()
    .str.replace("-", "", regex=False)
    .str.replace(" ", "", regex=False)
)

df[["contact_method_raw", "contact_method_step1"]].drop_duplicates().sort_values("contact_method_raw")

,contact_method_raw,contact_method_step1
8,Chat,chat
10,E-Mail,email
6,EMAIL,email
0,Email,email
3,Phone Call,phonecall
4,SMS,sms
11,Text Message,textmessage
7,chat,chat
1,e-mail,email
9,email,email


In [23]:
contact_method_map = {
    "email": "Email",
    "phone": "Phone",
    "phonecall": "Phone",
    "sms": "Text",
    "text": "Text",
    "textmessage": "Text",
    "chat": "Chat"
}

df["contact_method_clean"] = df["contact_method_step1"].map(contact_method_map)

df[["contact_method_raw", "contact_method_clean"]].drop_duplicates().sort_values("contact_method_raw")

,contact_method_raw,contact_method_clean
8,Chat,Chat
10,E-Mail,Email
6,EMAIL,Email
0,Email,Email
3,Phone Call,Phone
4,SMS,Text
11,Text Message,Text
7,chat,Chat
1,e-mail,Email
9,email,Email


## Part 6: Recode Customer Segments into Business Categories

Sometimes we keep detailed labels. Sometimes we group them into broader categories for analysis.

Here, we will recode customer segments into three broad groups:

- `High Value`: Premium or VIP
- `Standard`: Standard or Basic
- `Student`: Student
- `Unknown`: Unknown or not classified

This is not just a technical step. It changes how the data will be interpreted.


In [14]:
df["segment_step1"] = df["segment_raw"].str.strip().str.lower()

segment_map = {
    "premium": "High Value",
    "vip": "High Value",
    "standard": "Standard",
    "basic": "Standard",
    "student": "Student",
    "unknown": "Unknown"
}

df["segment_group"] = df["segment_step1"].map(segment_map)

df[["segment_raw", "segment_group"]].drop_duplicates().sort_values("segment_raw")

,segment_raw,segment_group
0,Premium,High Value
8,Basic,Standard
6,Premium,High Value
4,Standard,Standard
13,Student,Student
1,VIP,High Value
3,basic,Standard
11,premium,High Value
2,standard,Standard
12,student,Student


In [15]:
df["segment_group"].value_counts(dropna=False)

segment_group
High Value    6
Standard      6
Student       2
Unknown       1
Name: count, dtype: int64

### Reflection

What assumption did we make when we grouped `premium` and `vip` together as `High Value`?

Write 1–2 sentences.


> Your answer here:
>1. We are assuming that Premium and VIP are close enough to treat as one group or tier.
>2. We are also assuming that these customers are more valuable than standard or other grouped customers.

## Part 7: Use `.str.contains()` to Create Flags

Text fields often contain useful signals. For example, the `issue_text` field may mention fraud or unauthorized charges.

We can create a simple indicator variable using `.str.contains()`.


In [25]:
df["possible_fraud_issue"] = df["issue_text"].str.contains(
    "fraud|unauthorized",
    case=False,
    na=False
)

df[["issue_text", "possible_fraud_issue"]].head(15)

,issue_text,possible_fraud_issue
0,Customer says login failed twice,False
1,Asked about mortgage rate,False
2,Reported unauthorized charge,True
3,Needs card replacement,False
4,Asked to update address,False
5,Possible fraud on account,True
6,Wants payoff estimate,False
7,Asked about late fee,False
8,Fraud alert question,True
9,New account opening,False


This flag does not prove fraud. It only identifies records where the text contains certain words. That distinction matters.

## Part 8: Use `.str.split()` for Simple Parsing

Sometimes one column contains multiple pieces of information. We can use `.str.split()` to separate simple combined fields.

Here, we will split the cleaned customer name into first and last names. This is just a demonstration. Real names can be more complicated than this, so this won't work perfectly in every situation.


In [26]:
df[["first_name", "last_name"]] = df["customer_name_clean"].str.split(" ", n=1, expand=True)

df[["customer_name_clean", "first_name", "last_name"]].head(10)

,customer_name_clean,first_name,last_name
0,Alice Smith,Alice,Smith
1,Bob Jones,Bob,Jones
2,Carla Nguyen,Carla,Nguyen
3,Daniel Lee,Daniel,Lee
4,Erin O'Neill,Erin,O'Neill
5,Fatima Khan,Fatima,Khan
6,George Martin,George,Martin
7,Hana Kim,Hana,Kim
8,Ivan Petrov,Ivan,Petrov
9,Julia Brown,Julia,Brown


## Part 9: Clean Product Categories

The `product_raw` column contains product names with underscores, inconsistent capitalization, and spacing. We will standardize these labels into readable product categories.


In [27]:
df["product_step1"] = (
    df["product_raw"]
    .str.strip()
    .str.lower()
    .str.replace("_", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
)

df[["product_raw", "product_step1"]].drop_duplicates().sort_values("product_raw")

,product_raw,product_step1
7,Auto_Loan,auto loan
2,Checking,checking
9,Checking_Account,checking account
11,Credit Card,credit card
3,Credit_Card,credit card
0,Loan_Application,loan application
5,Mortgage,mortgage
14,Personal_Loan,personal loan
12,Student_Loan,student loan
8,auto loan,auto loan


In [28]:
product_map = {
    "loan application": "Loan Application",
    "checking": "Checking Account",
    "checking account": "Checking Account",
    "credit card": "Credit Card",
    "mortgage": "Mortgage",
    "auto loan": "Auto Loan",
    "student loan": "Student Loan",
    "personal loan": "Personal Loan"
}

df["product_clean"] = df["product_step1"].map(product_map)

df[["product_raw", "product_clean"]].drop_duplicates().sort_values("product_raw")

,product_raw,product_clean
7,Auto_Loan,Auto Loan
2,Checking,Checking Account
9,Checking_Account,Checking Account
11,Credit Card,Credit Card
3,Credit_Card,Credit Card
0,Loan_Application,Loan Application
5,Mortgage,Mortgage
14,Personal_Loan,Personal Loan
12,Student_Loan,Student Loan
8,auto loan,Auto Loan


## Part 10: Review the Guided Transformations

Before the independent exercises, create a compact review table from the guided transformations. We keep `issue_text` temporarily so we can compare it with the keyword flag. This is a working review table, not the final file for broad analysis.

In [29]:
guided_review = df[[
    "customer_id",
    "state_clean",
    "segment_group",
    "contact_method_clean",
    "product_clean",
    "possible_fraud_issue",
    "issue_text",
]].copy()

guided_review.head(10)

,customer_id,state_clean,segment_group,contact_method_clean,product_clean,possible_fraud_issue,issue_text
0,1001,TX,High Value,Email,Loan Application,False,Customer says login failed twice
1,1002,TX,High Value,Email,Loan Application,False,Asked about mortgage rate
2,1003,TX,Standard,Phone,Checking Account,True,Reported unauthorized charge
3,1004,CA,Standard,Phone,Credit Card,False,Needs card replacement
4,1005,CA,Standard,Text,Credit Card,False,Asked to update address
5,1006,CA,High Value,Text,Mortgage,True,Possible fraud on account
6,1007,NY,High Value,Email,Mortgage,False,Wants payoff estimate
7,1008,NY,Unknown,Chat,Auto Loan,False,Asked about late fee
8,1009,NY,Standard,Chat,Auto Loan,True,Fraud alert question
9,1010,FL,Standard,Email,Checking Account,False,New account opening


## Part 11: Short Recoding Decision Log

Choose two transformations from state, contact method, segment, or product. For each one, record the raw labels combined, the resulting category, why the decision is reasonable for this lab, and one detail or distinction that could be lost.

### Recoding Decision Log

**Decision 1**

- Field and raw labels: contact_method_raw: Different variations of 'Text' category including SMS and Text variants
- Resulting category: Text
- Reason: Text is a direct reflection of the values categorized. 
- Caveat or detail lost: We lost 'SMS' from the raw data. If this assumption is incorrect then we will need to keep SMS in a different category from text.

**Decision 2**

- Field and raw labels: product_raw: Checking_Account, checking, checking account
- Resulting category: Checking Account
- Reason: combined all these into a standard category by removing underscores and capitalizing each word. 
- Caveat or detail lost: If the different labels represented a differenct type of checking account for example a high interest earning checking vs a basic checking then that will be lost by combining them into the resulting category. 

## Part 12: Your Turn

Complete the exercises below using the same dataset.

Use only the Pandas tools introduced in this lab.


### Exercise 1: Standardize Channel Labels

Create a new column called `channel_clean`.

Suggested categories:

- `Web Organic`
- `Web Paid`
- `Mobile App`
- `Branch`
- `Call Center`
- `Partner Referral`
- `Unknown`

Hint: start by using `.str.strip()`, `.str.lower()`, and `.str.replace()`.


In [32]:
# Your code here
df["channel_step1"] = (
    df["channel_raw"]
    .str.strip()
    .str.lower()
    .str.replace("-", "", regex=False)
    .str.replace(" ", "", regex=False)
)

channel_map = {
    "weborganic": "Web Organic",
    "webpaid": "Web Paid",
    "mobileapp": "Mobile App",
    "branch": "Branch",
    "callcenter": "Call Center",
    "partnerreferral": "Partner Referral",
    "unknown": "Unknown"
}

df["channel_clean"] = df["channel_step1"].map(channel_map)

df[["channel_raw", "channel_step1", "channel_clean"]].drop_duplicates().sort_values("channel_raw")

,channel_raw,channel_step1,channel_clean
11,Branch,branch,Branch
7,Call Center,callcenter,Call Center
2,Mobile App,mobileapp,Mobile App
9,Partner Referral,partnerreferral,Partner Referral
14,Unknown,unknown,Unknown
5,WEB - PAID,webpaid,Web Paid
1,Web-Organic,weborganic,Web Organic
3,branch,branch,Branch
8,call center,callcenter,Call Center
4,mobile-app,mobileapp,Mobile App


### Exercise 2: Create a Digital Channel Flag

Create a Boolean column called `digital_channel`.

Set it to `True` when the cleaned channel is one of:

- `Web Organic`
- `Web Paid`
- `Mobile App`

Otherwise, set it to `False`.

Hint: use `.isin()`.


In [33]:
# Your code here
digital_channels = ["Web Organic", "Web Paid", "Mobile App"]
df["digital_channel"] = df["channel_clean"].isin(digital_channels)
df[["channel_raw", "channel_clean", "digital_channel"]].drop_duplicates().sort_values("channel_clean")

,channel_raw,channel_clean,digital_channel
3,branch,Branch,False
11,Branch,Branch,False
7,Call Center,Call Center,False
8,call center,Call Center,False
2,Mobile App,Mobile App,True
4,mobile-app,Mobile App,True
9,Partner Referral,Partner Referral,False
10,partner-referral,Partner Referral,False
14,Unknown,Unknown,False
0,web - organic,Web Organic,True


### Exercise 3: Create State Region Categories

Create a new column called `state_region` using `state_clean`.

Suggested mapping:

- `TX`, `FL` → `South`
- `CA`, `NV` → `West`
- `NY` → `Northeast`

Use `.map()`.


In [34]:
# Your code here
region_map = {
    "TX": "South",
    "FL": "South",
    "CA": "West",
    "NV": "West",
    "NY": "Northeast"
}

df["state_region"] = df["state_clean"].map(region_map)
df[["state_raw", "state_clean", "state_region"]].drop_duplicates().sort_values("state_clean")

,state_raw,state_clean,state_region
3,CA,CA,West
4,California,CA,West
5,ca,CA,West
9,FL,FL,South
10,Florida,FL,South
11,fl.,FL,South
14,Nevada,NV,West
6,NY,NY,Northeast
7,New York,NY,Northeast
8,N.Y.,NY,Northeast


## Part 13: Build, Verify, and Export the Analysis Dataset

The final output uses the three columns created in Exercises 1-3. It excludes names, email addresses, and free-text issue details because they are not needed for the intended channel-and-region analysis.

Run the next cell after completing the exercises. Review the row count and output column list before saving the file.


In [35]:
ready_to_export = (
    "channel_clean" in df.columns
    and "digital_channel" in df.columns
    and "state_region" in df.columns
)

if not ready_to_export:
    print("Complete Exercises 1-3 before exporting the analysis dataset.")
else:
    analysis_df = df[[
        "customer_id",
        "state_clean",
        "state_region",
        "segment_group",
        "contact_method_clean",
        "channel_clean",
        "digital_channel",
        "product_clean",
        "possible_fraud_issue",
    ]].copy()

    print("Rows preserved:", len(analysis_df) == len(df_raw))
    print("Output columns:", analysis_df.columns.tolist())

    output_dir = Path("../data/processed")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / "customer_contacts_transformed_module6.csv"
    analysis_df.to_csv(output_path, index=False)

    print(f"Saved {len(analysis_df)} rows to: {output_path}")
    display(analysis_df.head())


Rows preserved: True
Output columns: ['customer_id', 'state_clean', 'state_region', 'segment_group', 'contact_method_clean', 'channel_clean', 'digital_channel', 'product_clean', 'possible_fraud_issue']
Saved 15 rows to: ..\data\processed\customer_contacts_transformed_module6.csv


,customer_id,state_clean,state_region,segment_group,contact_method_clean,channel_clean,digital_channel,product_clean,possible_fraud_issue
0,1001,TX,South,High Value,Email,Web Organic,True,Loan Application,False
1,1002,TX,South,High Value,Email,Web Organic,True,Loan Application,False
2,1003,TX,South,Standard,Phone,Mobile App,True,Checking Account,True
3,1004,CA,West,Standard,Phone,Branch,False,Credit Card,False
4,1005,CA,West,Standard,Text,Mobile App,True,Credit Card,False


## Part 14: Offline Assignment - Recoding Fitness Check

Scenario: a marketing analyst wants to compare digital contact activity by broad region. Before approving the transformed file for that purpose:

1. Display the distinct mapping from `channel_raw` to `channel_clean` and `digital_channel`.
2. Display the distinct mapping from `state_raw` to `state_clean` and `state_region`.
3. Count missing values in `channel_clean`, `digital_channel`, `state_clean`, and `state_region`.
4. Decide whether the mappings are fit for this specific analysis.

In [39]:
# Offline Assignment - Your code here
# Hints:
# - Select each raw/clean mapping pair and use .drop_duplicates().
# - Sort the mapping review by the raw value so it is easy to scan.
# - Select the four cleaned mapping columns, then use .isna().sum().
# Create channel_mapping_review, state_mapping_review, and missing_mapping_counts.
# Display all three results.

#1. channel mapping
channel_mapping_review = (df[["channel_raw", "channel_clean", "digital_channel"]].drop_duplicates().sort_values("channel_raw"))

#2. state mapping
state_mapping_review = (df[["state_raw", "state_clean", "state_region"]].drop_duplicates().sort_values("state_raw"))

#3. missing mapping counts
missing_mapping_counts = (df[["channel_clean", "digital_channel", "state_clean", "state_region"]].isna().sum())

display(channel_mapping_review, state_mapping_review, missing_mapping_counts)

,channel_raw,channel_clean,digital_channel
11,Branch,Branch,False
7,Call Center,Call Center,False
2,Mobile App,Mobile App,True
9,Partner Referral,Partner Referral,False
14,Unknown,Unknown,False
5,WEB - PAID,Web Paid,True
1,Web-Organic,Web Organic,True
3,branch,Branch,False
8,call center,Call Center,False
4,mobile-app,Mobile App,True


,state_raw,state_clean,state_region
3,CA,CA,West
4,California,CA,West
9,FL,FL,South
10,Florida,FL,South
8,N.Y.,NY,Northeast
6,NY,NY,Northeast
14,Nevada,NV,West
7,New York,NY,Northeast
12,TX,TX,South
0,TX,TX,South


channel_clean      0
digital_channel    0
state_clean        0
state_region       0
dtype: int64

### Fitness Assessment

**Your response here:** Write 4-5 sentences. Cite at least two results from your checks, explain one business assumption behind the digital or region grouping, identify one detail lost through recoding, and state whether you would approve these mappings for the requested analysis.

- The mapping checks show that 13 distinct raw channel variants have been recoded to the clean channel category.
- The mapping checks show that 14 distinct raw state variants have been recoded to the clean state category.
- The state grouping assumes there is no need to distinguish by state in analysis. Texas and Florida are both grouped as South but have many differences.
- A detail lost through recoding is the grouping of Web Paid, Web Organic, and Mobile App into a digital channel. Depending on the business needs it may be important to distinguish if digital activity was bought or earned organically. 
- I would approve thes emappings if the intention is for a broader view or analysis perhaps for a presentation at a high level to stakeholders. 

## Final Reminder

String and categorical transformations affect counts, summaries, groups, dashboards, and model features. Before submitting:

- complete Exercises 1-3;
- complete both recoding decision-log entries;
- confirm that no expected cleaned category is missing;
- export `data/processed/customer_contacts_transformed_module6.csv`;
- complete the offline assignment and written assessment; and
- rerun the notebook from the top and save it.

Before any recoding decision, ask what labels are truly equivalent, what meaning might be lost, and whether the rule can be explained to another analyst.